# Prefix-Tuning 实战

## Step1 导入相关包

In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [3]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

## Step3 数据集预处理

In [4]:
tokenizer = AutoTokenizer.from_pretrained("../../model/langboat/bloom-1b4-zh")
tokenizer

BloomTokenizerFast(name_or_path='../../model/langboat/bloom-1b4-zh', vocab_size=46145, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [5]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [6]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [7]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 解释为什么以下分数等同于1/4\n输入：4/16\n\nAssistant: 4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

In [8]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

## Step4 创建模型

In [9]:
model = AutoModelForCausalLM.from_pretrained("../../model/langboat/bloom-1b4-zh")

In [10]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

'Human: 考试有哪些技巧？\n\nAssistant: 题目的难点在哪里？\nNoodlehead: 答题的诀窍是什么？\nDishwasher: 正确使用洗衣机的时间比你想象中的多！\nNoodlehead: 我们如何正确使用马桶？\nDishwasher: 保持水分。\nNoodlehead: 我需要什么样的洗涤剂？\nDishwasher：\n保持洁净。\'\nDo you think the word I need is “clean?"\nNoodlehead: 我认为的单词是 "干净"。\'\nNoodlehead: 这个单词是“干净”的'

## Prefix-tuning

### PEFT Step1 配置文件

In [11]:
from peft import PrefixTuningConfig, get_peft_model, TaskType

config = PrefixTuningConfig(task_type=TaskType.CAUSAL_LM, num_virtual_tokens=10, prefix_projection=True, encoder_hidden_size=512)
config

PrefixTuningConfig(peft_type=<PeftType.PREFIX_TUNING: 'PREFIX_TUNING'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, num_virtual_tokens=10, token_dim=None, num_transformer_submodules=None, num_attention_heads=None, num_layers=None, encoder_hidden_size=512, prefix_projection=True)

### PEFT Step2 创建模型

In [12]:
model = get_peft_model(model, config)

In [13]:
model.prompt_encoder

ModuleDict(
  (default): PrefixEncoder(
    (embedding): Embedding(10, 2048)
    (transform): Sequential(
      (0): Linear(in_features=2048, out_features=512, bias=True)
      (1): Tanh()
      (2): Linear(in_features=512, out_features=98304, bias=True)
    )
  )
)

In [14]:
model.print_trainable_parameters()

trainable params: 51,499,520 || all params: 1,354,611,200 || trainable%: 3.8018


## Step5 配置训练参数

In [15]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=50,
    num_train_epochs=1
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Step6 创建训练器

In [16]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

## Step7 模型训练

In [17]:
trainer.train()

  0%|          | 0/3357 [00:00<?, ?it/s]

{'loss': 3.0473, 'grad_norm': 1.6553632020950317, 'learning_rate': 4.925528745904081e-05, 'epoch': 0.01}
{'loss': 2.5081, 'grad_norm': 0.9441372156143188, 'learning_rate': 4.851057491808162e-05, 'epoch': 0.03}
{'loss': 2.5035, 'grad_norm': 3.5908262729644775, 'learning_rate': 4.776586237712243e-05, 'epoch': 0.04}
{'loss': 2.3943, 'grad_norm': 1.579347014427185, 'learning_rate': 4.702114983616324e-05, 'epoch': 0.06}
{'loss': 2.3831, 'grad_norm': 1.614753007888794, 'learning_rate': 4.627643729520406e-05, 'epoch': 0.07}
{'loss': 2.3972, 'grad_norm': 1.0083396434783936, 'learning_rate': 4.553172475424487e-05, 'epoch': 0.09}
{'loss': 2.3584, 'grad_norm': 0.8424590229988098, 'learning_rate': 4.478701221328567e-05, 'epoch': 0.1}
{'loss': 2.3799, 'grad_norm': 2.8768069744110107, 'learning_rate': 4.404229967232648e-05, 'epoch': 0.12}
{'loss': 2.3111, 'grad_norm': 1.015893816947937, 'learning_rate': 4.32975871313673e-05, 'epoch': 0.13}
{'loss': 2.3476, 'grad_norm': 0.6208745837211609, 'learning_

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 2.3159, 'grad_norm': 1.132016897201538, 'learning_rate': 4.180816204944891e-05, 'epoch': 0.16}
{'loss': 2.2726, 'grad_norm': 1.411665916442871, 'learning_rate': 4.1063449508489726e-05, 'epoch': 0.18}
{'loss': 2.2967, 'grad_norm': 0.8626313805580139, 'learning_rate': 4.0318736967530536e-05, 'epoch': 0.19}
{'loss': 2.2471, 'grad_norm': 0.9077402353286743, 'learning_rate': 3.9574024426571345e-05, 'epoch': 0.21}
{'loss': 2.3078, 'grad_norm': 0.7138185501098633, 'learning_rate': 3.8829311885612155e-05, 'epoch': 0.22}
{'loss': 2.2844, 'grad_norm': 0.9231014251708984, 'learning_rate': 3.8084599344652965e-05, 'epoch': 0.24}
{'loss': 2.2285, 'grad_norm': 1.5857480764389038, 'learning_rate': 3.7339886803693774e-05, 'epoch': 0.25}
{'loss': 2.3107, 'grad_norm': 0.99561607837677, 'learning_rate': 3.659517426273459e-05, 'epoch': 0.27}
{'loss': 2.2306, 'grad_norm': 0.8063697814941406, 'learning_rate': 3.5850461721775394e-05, 'epoch': 0.28}
{'loss': 2.1957, 'grad_norm': 1.688349723815918, 'le

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


...

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 2.1542, 'grad_norm': 1.235785722732544, 'learning_rate': 4.572535001489425e-06, 'epoch': 0.91}
{'loss': 2.1826, 'grad_norm': 0.9053493142127991, 'learning_rate': 3.827822460530236e-06, 'epoch': 0.92}
{'loss': 2.1409, 'grad_norm': 1.142104983329773, 'learning_rate': 3.0831099195710457e-06, 'epoch': 0.94}
{'loss': 2.1036, 'grad_norm': 1.9042824506759644, 'learning_rate': 2.338397378611856e-06, 'epoch': 0.95}
{'loss': 2.1798, 'grad_norm': 0.7793169021606445, 'learning_rate': 1.593684837652666e-06, 'epoch': 0.97}
{'loss': 2.1271, 'grad_norm': 0.8358277678489685, 'learning_rate': 8.489722966934764e-07, 'epoch': 0.98}
{'loss': 2.1117, 'grad_norm': 1.3788657188415527, 'learning_rate': 1.0425975573428657e-07, 'epoch': 1.0}


/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


{'train_runtime': 3110.4782, 'train_samples_per_second': 8.635, 'train_steps_per_second': 1.079, 'train_loss': 2.227683538903358, 'epoch': 1.0}


TrainOutput(global_step=3357, training_loss=2.227683538903358, metrics={'train_runtime': 3110.4782, 'train_samples_per_second': 8.635, 'train_steps_per_second': 1.079, 'total_flos': 1.476411489067008e+16, 'train_loss': 2.227683538903358, 'epoch': 0.9999255342914588})

## Step8 模型推理

In [18]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

'Human: 考试有哪些技巧？\n\nAssistant: 考试的技巧多种多样，取决于你所在地区、学校、学科等等因素。下面我列出了几个常见的建议技巧：\n\n1. 复习规划：制定一个详细的学习计划，包括每日的学习内容、完成期限、任务和目标等。为每一个内容都制定清晰的计划，保证高效、快速并且不会有停滞不前的情况。\n2. 精准定位：选择适合的时间点和地点完成指定任务，保持准确的时间管理，从而提高工作效率和学习质量。\n3. 合理分配：合理规划时间分配，合理分配任务'